# Repository Ingestion to ChromaDB (Code RAG)

This notebook clones the public repository **iamgenii** and loads its source files into a local Chroma vector database for semantic search and Retrieval-Augmented Generation (RAG).

### Workflow:
1. **Clone Repository**: Fetch the code from the Git repository.
2. **Scan & Parse Files**: Find all source code and text files, ignoring binaries and configuration files.
3. **Chunk Content**: Split large files into overlapping line-based chunks to fit within the vector model's context window.
4. **Ingest to ChromaDB**: Set up a local Chroma DB instance and populate it with code snippets.
5. **Search Query**: Execute test semantic searches to retrieve relevant parts of the code.

In [3]:
# Ensure chromadb is installed in the active notebook kernel environment

import sys
import subprocess
import chromadb

import os
import glob
from pathlib import Path

In [4]:
# Configure repository details
REPO_URL = "https://github.com/codagelabs/iamgenii.git"
CLONE_DIR = "./iamgenii"

# Clone if it doesn't already exist
if not os.path.exists(CLONE_DIR):
    print(f"Cloning {REPO_URL} into {CLONE_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], check=True)
    print("Cloning complete!")
else:
    print(f"Directory '{CLONE_DIR}' already exists. Skipping clone.")

Cloning https://github.com/codagelabs/iamgenii.git into ./iamgenii...


Cloning into './iamgenii'...


Cloning complete!


In [5]:
# Allowed file extensions for ingestion
ALLOWED_EXTENSIONS = {
    '.go', '.py', '.md', '.sql', '.yaml', '.yml', 
    '.json', '.txt', '.html', '.css', '.js', '.sh'
}

ALLOWED_FILENAMES = {
    'Dockerfile', 'Makefile', 'go.mod', 'go.sum', 'Jenkinsfile'
}

def chunk_file_content(content, max_lines=40, overlap=10):
    """Splits a file content into overlapping chunks of lines."""
    lines = content.split('\n')
    chunks = []
    num_lines = len(lines)
    start = 0
    while start < num_lines:
        end = min(start + max_lines, num_lines)
        chunk_text = '\n'.join(lines[start:end])
        chunks.append({
            "content": chunk_text,
            "start_line": start + 1,
            "end_line": end
        })
        if end == num_lines:
            break
        start += max_lines - overlap
    return chunks

# Scan and read files
documents = []
metadatas = []
ids = []

print(f"Scanning directory: {CLONE_DIR}")
clone_path = Path(CLONE_DIR)

for path in clone_path.rglob('*'):
    # Ignore git internals, hidden folders, and directories
    if not path.is_file() or any(part.startswith('.') for part in path.parts):
        continue
        
    # Filter by extensions or specific filenames
    if path.suffix in ALLOWED_EXTENSIONS or path.name in ALLOWED_FILENAMES:
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                
            # Chunk the file content
            chunks = chunk_file_content(content)
            
            # Generate chunk documents
            relative_path = os.path.relpath(path, CLONE_DIR)
            for i, chunk in enumerate(chunks):
                chunk_id = f"{relative_path}#L{chunk['start_line']}-L{chunk['end_line']}"
                documents.append(chunk.get("content"))
                metadatas.append({
                    "source": relative_path,
                    "filename": path.name,
                    "file_type": path.suffix or path.name,
                    "start_line": chunk["start_line"],
                    "end_line": chunk["end_line"]
                })
                ids.append(chunk_id)
        except Exception as e:
            print(f"Error reading {path}: {e}")

print(f"Found and parsed {len(documents)} chunks from {len(set(m['source'] for m in metadatas))} files.")

Scanning directory: ./iamgenii
Found and parsed 410 chunks from 135 files.


In [6]:
# Set up persistent ChromaDB client
DB_PATH = "./chroma_db"
print(f"Initializing ChromaDB at {DB_PATH}...")
client = chromadb.PersistentClient(path=DB_PATH)

COLLECTION_NAME = "iamgenii_code"

# Reset collection if exists to avoid duplication
if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    print(f"Collection '{COLLECTION_NAME}' already exists. Re-creating...")
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(name=COLLECTION_NAME)

print(f"Ingesting {len(documents)} chunks into collection '{COLLECTION_NAME}'...")

# Batch ingestion
batch_size = 50
for i in range(0, len(documents), batch_size):
    batch_ids = ids[i : i + batch_size]
    batch_docs = documents[i : i + batch_size]
    batch_metas = metadatas[i : i + batch_size]
    
    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_metas
    )
    print(f"Ingested batch {i // batch_size + 1}/{(len(documents) - 1) // batch_size + 1}")

print(f"\nSuccessfully loaded {collection.count()} chunks into Chroma collection '{COLLECTION_NAME}'")

Initializing ChromaDB at ./chroma_db...
Ingesting 410 chunks into collection 'iamgenii_code'...
Ingested batch 1/9
Ingested batch 2/9
Ingested batch 3/9
Ingested batch 4/9
Ingested batch 5/9
Ingested batch 6/9
Ingested batch 7/9
Ingested batch 8/9
Ingested batch 9/9

Successfully loaded 410 chunks into Chroma collection 'iamgenii_code'


In [ ]:
# Run semantic queries to test search
test_queries = [
    "Database connection or schema configuration",
]

for query in test_queries:
    print(f"\n{'='*50}\nQuery: '{query}'\n{'='*50}")
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    
    for idx, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
        print(f"\nResult {idx + 1} (Distance: {dist:.4f}):")
        print(f"Source File: {meta['source']} (Lines {meta['start_line']}-{meta['end_line']})")
        print("-" * 30)
        print(doc)
        print("-" * 30)


Query: 'Database connection or schema configuration'

Result 1 (Distance: 1.0123):
Source File: database/mysql.go (Lines 1-35)
------------------------------
package database

import (
	//mysql is for mysql connection driver
	"time"

	"github.com/iamgenii/configs"
	_ "github.com/go-sql-driver/mysql"
	"github.com/jinzhu/gorm"
)

// DataStore defines database connection
// here we are using gorm for connection
type DataStore struct {
	db *gorm.DB
}

// NewDataStore for create mysql database connection
// get and connection string as parameter
// client connection and return a connection
func NewDataStore(connectionStr string, dbConfig configs.DBConfig) (*gorm.DB, error) {

	//Open connection using gorm
	client, err := gorm.Open("mysql", connectionStr)
	if err != nil {
		panic("Error in establish database client connection")
	}
	client.LogMode(dbConfig.DBLogMode)
	client.DB().SetMaxIdleConns(dbConfig.DbConnectionPool.MaxIdealConnection)
	client.DB().SetConnMaxLifetime(time.Duration(dbCon